# L02 -- Distribution Network Power Flow

Companion notebook for L02. After this notebook you should:
- Build a 4-bus radial feeder in pandapower
- Reproduce the BFS voltage profile by hand in numpy
- Compare LinDistFlow to the exact BFS result
- Demonstrate end-of-feeder PV voltage rise

**Prerequisite**: L01 notebook + slides `L02_distribution_network_pf.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pandapower as pp
import matplotlib.pyplot as plt

print(f"pandapower {pp.__version__}, numpy {np.__version__}, pandas {pd.__version__}")
assert pp.__version__ >= "3.2", "Need pandapower >= 3.2 (env: pdpower)"

## 1. Build the 4-bus radial feeder

Slack at bus 1; loads `S_2=0.10+j0.04`, `S_3=0.15+j0.06`, `S_4=0.20+j0.08` pu on a 1 MVA / 0.4 kV base.
Line impedances:
- $z_{12} = 0.02 + j0.04$,  $z_{23} = 0.03 + j0.05$,  $z_{34} = 0.04 + j0.06$ (pu)

In [ ]:
ZB = 0.16  # Z_base = V^2 / S = 0.4^2 / 1 ohm

def build_l02_feeder():
    net = pp.create_empty_network(sn_mva=1.0)
    bs = [pp.create_bus(net, vn_kv=0.4, name=f"b{i+1}") for i in range(4)]
    pp.create_ext_grid(net, bus=bs[0], vm_pu=1.0)
    line_params = [(0.02, 0.04), (0.03, 0.05), (0.04, 0.06)]
    for i, (r, x) in enumerate(line_params):
        pp.create_line_from_parameters(net, bs[i], bs[i+1], length_km=1.0,
            r_ohm_per_km=r*ZB, x_ohm_per_km=x*ZB, c_nf_per_km=0.0, max_i_ka=10.0)
    pp.create_load(net, bus=bs[1], p_mw=0.10, q_mvar=0.04)
    pp.create_load(net, bus=bs[2], p_mw=0.15, q_mvar=0.06)
    pp.create_load(net, bus=bs[3], p_mw=0.20, q_mvar=0.08)
    return net

net = build_l02_feeder()
pp.runpp(net, algorithm="bfsw")
print("converged:", net.converged)
print(net.res_bus[["vm_pu", "va_degree"]].round(4))

## 2. One BFS iteration by hand

Following the slide step-by-step:
1. **Backward sweep**: load currents `I_i^load = conj(S_i / V_i)`, branch currents accumulate to root.
2. **Forward sweep**: `V_j = V_i - z_{ij} I_{ij}`.

We do *one* iteration from flat start to match the slide tabulated values.

In [ ]:
z = [0.02+0.04j, 0.03+0.05j, 0.04+0.06j]    # z_12, z_23, z_34 in pu
S = [0.10+0.04j, 0.15+0.06j, 0.20+0.08j]    # S_2, S_3, S_4 in pu
V = [1.0+0j, 1.0+0j, 1.0+0j, 1.0+0j]        # flat start

# Backward sweep
Iload = [np.conj(S[k] / V[k+1]) for k in range(3)]   # currents at buses 2, 3, 4
Ibranch = [None, None, None]
Ibranch[2] = Iload[2]                                 # 3 -> 4
Ibranch[1] = Iload[1] + Ibranch[2]                    # 2 -> 3
Ibranch[0] = Iload[0] + Ibranch[1]                    # 1 -> 2
print("Branch currents (1->2, 2->3, 3->4):", [f"{i.real:.4f}{i.imag:+.4f}j" for i in Ibranch])

# Forward sweep
for k in range(3):
    V[k+1] = V[k] - z[k] * Ibranch[k]

for i, v in enumerate(V[1:], start=2):
    print(f"|V_{i}| = {abs(v):.4f}  V_{i} = {v.real:.4f}{v.imag:+.4f}j")

In [ ]:
# Compare hand-math iter-1 to pandapower's converged result
slide_vals = {2: 0.984, 3: 0.967, 4: 0.954}
for bus_idx, slide_v in slide_vals.items():
    hand_v = abs(V[bus_idx - 1])
    pp_v   = net.res_bus.vm_pu[bus_idx - 1]
    print(f"bus {bus_idx}: slide={slide_v:.3f}, hand={hand_v:.3f}, pp_converged={pp_v:.3f}")
    assert abs(hand_v - slide_v) < 2e-3, f"hand math disagrees with slide at bus {bus_idx}"
print("L02 hand-math agrees with slide values.")

## 3. LinDistFlow vs exact BFS

LinDistFlow drops the $|Z|^2 |I|^2$ loss term. On this feeder voltages are near 1 pu so it should be a near-perfect approximation.

In [ ]:
# LinDistFlow: |V_i - V_j| ~ R*P_{ij} + X*Q_{ij}
P_downstream = [0.10+0.15+0.20, 0.15+0.20, 0.20]   # P on branches 1->2, 2->3, 3->4
Q_downstream = [0.04+0.06+0.08, 0.06+0.08, 0.08]
V_lin = [1.0]
for k in range(3):
    dV = z[k].real * P_downstream[k] + z[k].imag * Q_downstream[k]
    V_lin.append(V_lin[-1] - dV)

print("bus  BFS     LinDist  diff")
for i in range(4):
    diff = abs(net.res_bus.vm_pu[i] - V_lin[i])
    print(f"{i+1:>3}  {net.res_bus.vm_pu[i]:.4f}  {V_lin[i]:.4f}  {diff:.4f}")

## 4. Voltage rise from end-of-feeder PV

Replace the 0.2 pu load at bus 4 with a 1.0 pu PV `sgen`. Net injection at bus 4 becomes $-0.8$ pu (after the original load is removed) and we should see $|V_4| > 1$.

In [ ]:
net_pv = build_l02_feeder()
# Drop the load at bus 4 and replace with a PV sgen
net_pv.load.drop(net_pv.load[net_pv.load.bus == 3].index, inplace=True)
pp.create_sgen(net_pv, bus=3, p_mw=1.0, q_mvar=0.0)
pp.runpp(net_pv, algorithm="bfsw")
print(net_pv.res_bus[["vm_pu"]].round(4))

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([1,2,3,4], net.res_bus.vm_pu,     "o-", label="loads only")
ax.plot([1,2,3,4], net_pv.res_bus.vm_pu,  "s-", label="end-of-feeder PV (1.0 pu)")
ax.axhline(1.05, color="red", linestyle="--", alpha=0.5)
ax.axhline(0.95, color="red", linestyle="--", alpha=0.5)
ax.set_xlabel("bus"); ax.set_ylabel("$|V|$ (pu)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Homework

### Required
1. Sweep PV at bus 4 from 0 to 2.0 pu in steps of 0.1. Plot `|V_4|` vs PV size. Find the hosting capacity at which `|V_4|` crosses 1.05.
2. Multiply all line impedances by 5 (long feeder). Re-run BFS and report which bus violates first.
3. Implement BFS yourself in $\sim$25 lines of numpy and confirm $|V_4|$ matches pandapower to 4 decimals.

### Optional
- Try `algorithm="nr"` on the original feeder and the impedance-multiplied feeder. Where does NR struggle?

In [ ]:
# TODO Q1: PV sweep and hosting-capacity plot
# pv_sizes = np.arange(0, 2.01, 0.1)
# ...

# TODO Q2: long-feeder violation analysis
# ...

# TODO Q3: numpy BFS
# def bfs_solve(z_list, S_list, V_slack=1.0, tol=1e-8, max_iter=50): ...